# 03 — Gold Aggregate: Project Profitability & Utilisation Analytics
Builds six analytical Gold tables from the Silver layer.

| Gold Table | Description |
|---|---|
| gold_consultant_utilisation | Per-consultant billable hours, utilisation rate |
| gold_project_profitability | Per-engagement margin and overrun analysis |
| gold_client_revenue | Client-level revenue, headcount, engagement counts |
| gold_delivery_health | Aggregated at-risk and delayed engagement metrics |
| gold_weekly_trends | Weekly billed value and hours trends |
| gold_portfolio_scorecards | Practice-level portfolio performance |

In [ ]:
from pyspark.sql import functions as F

sc   = spark.table("silver_consultants")
scli = spark.table("silver_clients")
seng = spark.table("silver_engagements")
sts  = spark.table("silver_timesheets")


In [ ]:
# ── gold_consultant_utilisation ──────────────────────────────────────────────
gold_cu = (
    sts
    .groupBy("consultant_id")
    .agg(
        F.round(F.sum("hours_logged"), 1).alias("total_hours_logged"),
        F.round(F.sum(F.when(F.col("is_billable") == 1, F.col("hours_logged")).otherwise(0)), 1).alias("billable_hours"),
        F.round(F.sum("billed_value_gbp"), 2).alias("total_billed_gbp"),
        F.countDistinct("engagement_id").alias("engagements_worked"),
    )
    .withColumn("utilisation_rate_pct",
        F.round(F.col("billable_hours") / F.col("total_hours_logged") * 100, 1))
    .join(sc.select("consultant_id", "grade", "practice", "region", "grade_band", "daily_rate_gbp"),
          on="consultant_id", how="left")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_cu.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_consultant_utilisation")
print(f"gold_consultant_utilisation: {gold_cu.count():,}")

In [ ]:
# ── gold_project_profitability ───────────────────────────────────────────────
gold_pp = (
    seng
    .select(
        "engagement_id", "client_id", "practice", "status",
        "budget_gbp", "actual_spend_gbp", "margin_pct", "margin_band",
        "overrun_pct", "delivery_health_flag", "duration_days", "headcount"
    )
    .withColumn("_updated_at", F.current_timestamp())
)
gold_pp.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_project_profitability")
print(f"gold_project_profitability: {gold_pp.count():,}")

In [ ]:
# ── gold_client_revenue ──────────────────────────────────────────────────────
eng_by_client = (
    seng
    .groupBy("client_id")
    .agg(
        F.count("engagement_id").alias("total_engagements"),
        F.round(F.sum("actual_spend_gbp"), 2).alias("total_revenue_gbp"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct"),
        F.sum("delivery_health_flag").alias("at_risk_engagements"),
    )
)

gold_cr = (
    scli.select("client_id", "client_name", "tier", "industry", "region",
                "nps_score", "nps_band", "contract_value_gbp", "concentration_pct")
    .join(eng_by_client, on="client_id", how="left")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_cr.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_client_revenue")
print(f"gold_client_revenue: {gold_cr.count():,}")

In [ ]:
# ── gold_delivery_health ─────────────────────────────────────────────────────
gold_dh = (
    seng
    .groupBy("practice", "status", "margin_band")
    .agg(
        F.count("engagement_id").alias("engagements"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct"),
        F.round(F.avg("overrun_pct"), 2).alias("avg_overrun_pct"),
        F.sum("delivery_health_flag").alias("at_risk_count"),
    )
    .withColumn("_updated_at", F.current_timestamp())
)
gold_dh.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_delivery_health")
print(f"gold_delivery_health: {gold_dh.count():,}")

In [ ]:
# ── gold_weekly_trends ───────────────────────────────────────────────────────
gold_wt = (
    sts
    .groupBy("week_starting")
    .agg(
        F.round(F.sum("hours_logged"), 1).alias("total_hours"),
        F.round(F.sum(F.when(F.col("is_billable") == 1, F.col("hours_logged")).otherwise(0)), 1).alias("billable_hours"),
        F.round(F.sum("billed_value_gbp"), 2).alias("total_billed_gbp"),
        F.countDistinct("consultant_id").alias("active_consultants"),
    )
    .withColumn("utilisation_rate_pct",
        F.round(F.col("billable_hours") / F.col("total_hours") * 100, 1))
    .orderBy("week_starting")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_wt.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_weekly_trends")
print(f"gold_weekly_trends: {gold_wt.count():,}")

In [ ]:
# ── gold_portfolio_scorecards ────────────────────────────────────────────────
ts_practice = (
    sts.join(sc.select("consultant_id", "practice"), on="consultant_id", how="left")
    .groupBy("practice")
    .agg(
        F.round(F.sum("billed_value_gbp"), 2).alias("total_billed_gbp"),
        F.round(F.sum("hours_logged"), 1).alias("total_hours"),
        F.countDistinct("consultant_id").alias("consultant_count"),
    )
)

eng_practice = (
    seng
    .groupBy("practice")
    .agg(
        F.count("engagement_id").alias("engagements"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct"),
        F.sum("delivery_health_flag").alias("at_risk_engagements"),
    )
)

gold_ps = (
    ts_practice
    .join(eng_practice, on="practice", how="left")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_ps.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("gold_portfolio_scorecards")
print(f"gold_portfolio_scorecards: {gold_ps.count():,}")

In [ ]:
print("\n=== Gold Aggregate Complete ===")
gold_tables = [
    "gold_consultant_utilisation", "gold_project_profitability", "gold_client_revenue",
    "gold_delivery_health", "gold_weekly_trends", "gold_portfolio_scorecards"
]
for tbl in gold_tables:
    cnt = spark.table(tbl).count()
    print(f"  {tbl}: {cnt:,}")